Soil Agent: Kenya Soil DSS prototype

What it is

An AI agent that answers questions about soil conditions and gives management recommendations, using the soil maps produced by the workflow in notebook "Quantile_Forest_Regression" under notebooks. It is built with smolagents from Hugging Face, as a CodeAgent powered by Qwen2.5-Coder-32B-Instruct through Hugging Face Inference.

What it does

Answers questions in plain language, such as "What is the soil pH around Kitale?", "Should a farmer near Kakamega town apply lime?" or "How much of the area within 5 km of Malava has low organic carbon?"
Finds places by name, looks up soil conditions at a point or across an area, and explains what the values mean.
Gives recommendations based on the soil class and its interpretation, stating the confidence and whether the location is within the area the model represents well.

Current coverage

Area: Kakamega and Trans Nzoia counties
Properties: soil pH and soil organic carbon (SOC), 0–30 cm
Data: legacy GloSIS-harmonized soil data, Quantile Regression Forest predictions, and indicator classes from config/indicator_thresholds.csv

Author

Japheth Kimeu
Geospatial & AI Technical Lead
FAO Kenya, SoilFER project

Version: prototype v0.1, September 2026

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/drive/MyDrive/soil_dss'   # adjust if your repo folder is elsewhere
os.chdir(REPO)
sys.path.insert(0, REPO)
print(os.listdir('agent'), os.listdir('outputs'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['__init__.py', '__pycache__', 'soil_agent.py', 'soil_tools.py'] ['model', 'maps', 'indicators']


In [2]:
!pip install -q smolagents rasterio

In [3]:
import agent.soil_tools as st
print("new version:", "other_significant_classes" in open(st.__file__).read())

new version: True


In [4]:
import json
from agent.soil_tools import find_place, get_soil_at_location, summarize_area, get_class_definitions

print(json.dumps(find_place(name="Kitale"), indent=1))
print(json.dumps(get_soil_at_location(lat=0.28, lon=34.75), indent=1))
print(json.dumps(summarize_area(lat=1.02, lon=35.0, radius_km=5), indent=1))
print(json.dumps(get_class_definitions(property_name="ph"), indent=1))

{
 "query": "Kitale",
 "matches": [
  {
   "name": "Kitale, Saboti, Trans Nzoia, Kenya",
   "lat": 1.01808,
   "lon": 35.00024,
   "type": "town"
  }
 ]
}
{
 "location": {
  "lat": 0.28,
  "lon": 34.75
 },
 "depth": "0-30 cm",
 "properties": {
  "ph": {
   "summary": "5.0 pH units, very strongly acid, possibly strongly acid (medium confidence), likely between 4.8 and 5.3 pH units; the location is well represented by the sampled sites",
   "advice": [
    "Very strongly acid: Aluminium toxicity likely and P poorly available; liming strongly advised",
    "If it is strongly acid instead: Aluminium toxicity possible for sensitive crops; liming advised for maize and beans"
   ],
   "advice_text": "Very strongly acid: Aluminium toxicity likely and P poorly available; liming strongly advised. If it is strongly acid instead: Aluminium toxicity possible for sensitive crops; liming advised for maize and beans",
   "label": "Soil pH (water)",
   "unit": "pH units",
   "predicted_value": 5.0,
   

In [5]:
import os
print(os.listdir('agent'))

['__init__.py', '__pycache__', 'soil_agent.py', 'soil_tools.py']


In [6]:
from google.colab import userdata
from huggingface_hub import whoami
from agent.soil_agent import build_agent

token = userdata.get('HF_TOKEN')
print(whoami(token=token)['name'])      # should print: kimeujapheth

agent = build_agent(token=token)
agent.run("What is the soil pH around Kitale, and does it need liming?")

kimeujapheth


╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the soil pH around Kitale, and does it need liming?                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  kitale_location = find_place(name="Kitale")                                                                      
  print(kitale_location)                                                                                           
  kitale_summary = summarize_area(lat=kitale_location["matches"][0]["lat"],                                        
  lon=kitale_location["matches"][0]["lon"], radius_km=3)                                                           
  print(kitale_summary)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'query': 'Kitale', 'matches': [{'name': 'Kitale, Saboti, Trans Nzoia, Kenya', 'lat': 1.01808, 'lon': 35.00024, 
'type': 'town'}]}
{'center': {'lat': 1.01808, 'lon': 35.00024}, 'radius_km': 3.0, 'depth': '0-30 cm', 'properties': {'ph': 
{'summary': 'a mean of 5.4 pH units within 3 km, strongly acid on 78% and very strongly acid on 12% and moderately 
acid on 10% of the area; confidence is medium on 48%, low on 52% of the area, and 82% of the area is well 
represented by the sampled sites', 'advice': ['On 78% of the area (strongly acid): Aluminium toxicity possible for 
sensitive crops; liming advised for maize and beans'], 'advice_text': 'On 78% of the area (strongly acid): 
Aluminium toxicity possible for sensitive crops; liming advised for maize and beans', 'label': 'Soil pH (water)', 
'unit': 'pH units', 'mapped_area_km2': 28.3, 'mean_value': 5.4, 'min_max_value': [4.8, 5.8], 'dominant_class': 
'Strongly acid', 'dominant_class_interpretation': 'Aluminium toxicity possible for sensitive crops; liming advised 
for maize and beans', 'dominant_class_explanation': 'Exchangeable aluminium may be present and can limit root 
growth of sensitive crops such as maize and beans; phosphorus availability is reduced; raising pH improves 
fertilizer use efficiency', 'other_significant_classes': [], 'class_distribution': [{'class': 'Strongly acid', 
'class_range': '5.1 to <5.6 pH units', 'percent_of_area': 77.8}, {'class': 'Very strongly acid', 'class_range': 
'4.5 to <5.1 pH units', 'percent_of_area': 11.9}, {'class': 'Moderately acid', 'class_range': '5.6 to <6.1 pH 
units', 'percent_of_area': 10.3}], 'confidence_percent': {'high': 0.0, 'medium': 48.0, 'low': 52.0}, 
'percent_inside_area_of_applicability': 82.2, 'class_ranges_source': 'USDA Soil Survey Manual (Soil Science 
Division Staff 2017)', 'interpretation_source': 'project indicator table (config/indicator_thresholds.csv)'}, 
'soc_gkg': {'summary': 'a mean of 16.7 g/kg within 3 km, moderate on 77% and high on 23% of the area; confidence is
high on 11%, medium on 87%, low on 2% of the area, and 82% of the area is well represented by the sampled sites', 
'advice': ['On 77% of the area (moderate): Adequate; maintain with residue retention and organic inputs'], 
'advice_text': 'On 77% of the area (moderate): Adequate; maintain with residue retention and organic inputs', 
'label': 'Soil organic carbon', 'unit': 'g/kg', 'mapped_area_km2': 28.3, 'mean_value': 16.7, 'min_max_value': 
[12.8, 24.6], 'dominant_class': 'Moderate', 'dominant_class_interpretation': 'Adequate; maintain with residue 
retention and organic inputs', 'dominant_class_explanation': 'Enough organic matter to support soil structure, 
water retention and nutrient supply; continued organic inputs are needed to prevent decline under continuous 
cropping', 'other_significant_classes': [], 'class_distribution': [{'class': 'Moderate', 'class_range': '10 to <18 
g/kg', 'percent_of_area': 77.2}, {'class': 'High', 'class_range': '18 to <30 g/kg', 'percent_of_area': 22.8}], 
'confidence_percent': {'high': 10.8, 'medium': 87.3, 'low': 2.0}, 'percent_inside_area_of_applicability': 81.6, 
'class_ranges_source': 'Hazelton & Murphy (2016) Interpreting Soil Test Results', 'interpretation_source': 'project
indicator table (config/indicator_thresholds.csv)'}}}

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  kitale_ph_summary = kitale_summary["properties"]["ph"]["summary"]                                                
  kitale_ph_advice_text = kitale_summary["properties"]["ph"]["advice_text"]                                        
  kitale_ph_dominant_class_explanation = kitale_summary["properties"]["ph"]["dominant_class_explanation"]          
  kitale_ph_class_ranges_source = kitale_summary["properties"]["ph"]["class_ranges_source"]                        
                                                                                                                   
  kitale_soc_summary = kitale_summary["properties"]["soc_gkg"]["summary"]                                          
  kitale_soc_advice_text = kitale_summary["properties"]["soc_gkg"]["advice_text"]                                  
  kitale_soc_dominant_class_explanation = kitale_summary["properties"]["soc_gkg"]["dominant_class_explanation"]    
  kitale_soc_class_ranges_source = kitale_summary["properties"]["soc_gkg"]["class_ranges_source"]                  
                                                                                                                   
  final_answer(f"Soil pH around Kitale: {kitale_ph_summary}. This means that the soil is predominantly acidic,     
  which can limit root growth for sensitive crops and reduce phosphorus availability. Soil organic carbon around   
  Kitale: {kitale_soc_summary}. This means that the soil has adequate organic matter to support soil structure,    
  water retention, and nutrient supply. Recommendations: {kitale_ph_advice_text}. {kitale_soc_advice_text}. Class  
  ranges follow {kitale_ph_class_ranges_source} (pH) and {kitale_soc_class_ranges_source} (SOC).")                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Soil pH around Kitale: a mean of 5.4 pH units within 3 km, strongly acid on 78% and very strongly 
acid on 12% and moderately acid on 10% of the area; confidence is medium on 48%, low on 52% of the area, and 82% of
the area is well represented by the sampled sites. This means that the soil is predominantly acidic, which can 
limit root growth for sensitive crops and reduce phosphorus availability. Soil organic carbon around Kitale: a mean
of 16.7 g/kg within 3 km, moderate on 77% and high on 23% of the area; confidence is high on 11%, medium on 87%, 
low on 2% of the area, and 82% of the area is well represented by the sampled sites. This means that the soil has 
adequate organic matter to support soil structure, water retention, and nutrient supply. Recommendations: On 78% of
the area (strongly acid): Aluminium toxicity possible for sensitive crops; liming advised for maize and beans. On 
77% of the area (moderate): Adequate; maintain with residue retention and organic inputs. Class ranges follow USDA 
Soil Survey Manual (Soil Science Division Staff 2017) (pH) and Hazelton & Murphy (2016) Interpreting Soil Test 
Results (SOC).

[Step 3: Duration 7.15 seconds| Input tokens: 18,363 | Output tokens: 680]

'Soil pH around Kitale: a mean of 5.4 pH units within 3 km, strongly acid on 78% and very strongly acid on 12% and moderately acid on 10% of the area; confidence is medium on 48%, low on 52% of the area, and 82% of the area is well represented by the sampled sites. This means that the soil is predominantly acidic, which can limit root growth for sensitive crops and reduce phosphorus availability. Soil organic carbon around Kitale: a mean of 16.7 g/kg within 3 km, moderate on 77% and high on 23% of the area; confidence is high on 11%, medium on 87%, low on 2% of the area, and 82% of the area is well represented by the sampled sites. This means that the soil has adequate organic matter to support soil structure, water retention, and nutrient supply. Recommendations: On 78% of the area (strongly acid): Aluminium toxicity possible for sensitive crops; liming advised for maize and beans. On 77% of the area (moderate): Adequate; maintain with residue retention and organic inputs. Class range

In [7]:
agent.run("Should a farmer near Kakamega town apply lime? Also comment on organic carbon.")

╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ Should a farmer near Kakamega town apply lime? Also comment on organic carbon.                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Find the coordinates of Kakamega town                                                                          
  kakamega_location = find_place(name="Kakamega town")                                                             
  print(kakamega_location)                                                                                         
                                                                                                                   
  # Summarize the soil conditions in the area around Kakamega town                                                 
  if "matches" in kakamega_location:                                                                               
      kakamega_coords = kakamega_location["matches"][0]                                                            
      soil_summary = summarize_area(lat=kakamega_coords["lat"], lon=kakamega_coords["lon"], radius_km=3)           
      print(soil_summary)                                                                                          
  else:                                                                                                            
      print("Error finding Kakamega town:", kakamega_location["error"])                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'query': 'Kakamega town', 'matches': [{'name': 'Kakamega, Lurambi, Kakamega, 50100, Kenya', 'lat': 0.2833, 'lon': 
34.75, 'type': 'town'}]}
{'center': {'lat': 0.2833, 'lon': 34.75}, 'radius_km': 3.0, 'depth': '0-30 cm', 'properties': {'ph': {'summary': 'a
mean of 5.0 pH units within 3 km, very strongly acid on 79% and strongly acid on 21% of the area; confidence is 
medium on 98%, low on 2% of the area, and 90% of the area is well represented by the sampled sites', 'advice': ['On
79% of the area (very strongly acid): Aluminium toxicity likely and P poorly available; liming strongly advised'], 
'advice_text': 'On 79% of the area (very strongly acid): Aluminium toxicity likely and P poorly available; liming 
strongly advised', 'label': 'Soil pH (water)', 'unit': 'pH units', 'mapped_area_km2': 28.3, 'mean_value': 5.0, 
'min_max_value': [4.8, 5.4], 'dominant_class': 'Very strongly acid', 'dominant_class_interpretation': 'Aluminium 
toxicity likely and P poorly available; liming strongly advised', 'dominant_class_explanation': 'Aluminium toxicity
commonly restricts root growth of maize and beans; much of the phosphorus (including fertilizer P) is fixed and 
unavailable; calcium and magnesium are often low, so fertilizer responses are poor', 'other_significant_classes': 
[], 'class_distribution': [{'class': 'Very strongly acid', 'class_range': '4.5 to <5.1 pH units', 
'percent_of_area': 79.3}, {'class': 'Strongly acid', 'class_range': '5.1 to <5.6 pH units', 'percent_of_area': 
20.7}], 'confidence_percent': {'high': 0.2, 'medium': 98.2, 'low': 1.7}, 'percent_inside_area_of_applicability': 
90.1, 'class_ranges_source': 'USDA Soil Survey Manual (Soil Science Division Staff 2017)', 'interpretation_source':
'project indicator table (config/indicator_thresholds.csv)'}, 'soc_gkg': {'summary': 'a mean of 19.2 g/kg within 3 
km, high on 93% and moderate on 7% of the area; confidence is high on 4%, medium on 96% of the area, and 89% of the
area is well represented by the sampled sites', 'advice': ['On 93% of the area (high): Good; maintain current 
practices'], 'advice_text': 'On 93% of the area (high): Good; maintain current practices', 'label': 'Soil organic 
carbon', 'unit': 'g/kg', 'mapped_area_km2': 28.3, 'mean_value': 19.2, 'min_max_value': [14.0, 27.2], 
'dominant_class': 'High', 'dominant_class_interpretation': 'Good; maintain current practices', 
'dominant_class_explanation': 'Good structure, water holding and nutrient supply from organic matter, supporting 
active soil biology', 'other_significant_classes': [], 'class_distribution': [{'class': 'High', 'class_range': '18 
to <30 g/kg', 'percent_of_area': 92.8}, {'class': 'Moderate', 'class_range': '10 to <18 g/kg', 'percent_of_area': 
7.2}], 'confidence_percent': {'high': 4.3, 'medium': 95.6, 'low': 0.0}, 'percent_inside_area_of_applicability': 
88.8, 'class_ranges_source': 'Hazelton & Murphy (2016) Interpreting Soil Test Results', 'interpretation_source': 
'project indicator table (config/indicator_thresholds.csv)'}}}

Out: {'center': {'lat': 0.2833, 'lon': 34.75}, 'radius_km': 3.0, 'depth': '0-30 cm', 'properties': {'ph': 
{'summary': 'a mean of 5.0 pH units within 3 km, very strongly acid on 79% and strongly acid on 21% of the area; 
confidence is medium on 98%, low on 2% of the area, and 90% of the area is well represented by the sampled sites', 
'advice': ['On 79% of the area (very strongly acid): Aluminium toxicity likely and P poorly available; liming 
strongly advised'], 'advice_text': 'On 79% of the area (very strongly acid): Aluminium toxicity likely and P poorly
available; liming strongly advised', 'label': 'Soil pH (water)', 'unit': 'pH units', 'mapped_area_km2': 28.3, 
'mean_value': 5.0, 'min_max_value': [4.8, 5.4], 'dominant_class': 'Very strongly acid', 
'dominant_class_interpretation': 'Aluminium toxicity likely and P poorly available; liming strongly advised', 
'dominant_class_explanation': 'Aluminium toxicity commonly restricts root growth of maize

[Step 1: Duration 4.43 seconds| Input tokens: 5,046 | Output tokens: 198]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Extracting the relevant summaries and advice texts                                                             
  ph_summary = soil_summary['properties']['ph']['summary']                                                         
  ph_advice_text = soil_summary['properties']['ph']['advice_text']                                                 
  soc_summary = soil_summary['properties']['soc_gkg']['summary']                                                   
  soc_advice_text = soil_summary['properties']['soc_gkg']['advice_text']                                           
                                                                                                                   
  # Extracting the class ranges sources                                                                            
  ph_class_ranges_source = soil_summary['properties']['ph']['class_ranges_source']                                 
  soc_class_ranges_source = soil_summary['properties']['soc_gkg']['class_ranges_source']                           
                                                                                                                   
  # Preparing the final answer                                                                                     
  final_answer(f"A farmer near Kakamega town should apply lime. Soil pH around Kakamega town: {ph_summary}. This   
  indicates that aluminium toxicity is likely and phosphorus is poorly available, suggesting strong liming         
  advice. Soil organic carbon around Kakamega town: {soc_summary}. This is considered good, supporting active      
  soil biology and maintaining current practices. Recommendations: {ph_advice_text}. {soc_advice_text}. Class      
  ranges follow {ph_class_ranges_source} (pH) and {soc_class_ranges_source} (SOC).")                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: A farmer near Kakamega town should apply lime. Soil pH around Kakamega town: a mean of 5.0 pH units 
within 3 km, very strongly acid on 79% and strongly acid on 21% of the area; confidence is medium on 98%, low on 2%
of the area, and 90% of the area is well represented by the sampled sites. This indicates that aluminium toxicity 
is likely and phosphorus is poorly available, suggesting strong liming advice. Soil organic carbon around Kakamega 
town: a mean of 19.2 g/kg within 3 km, high on 93% and moderate on 7% of the area; confidence is high on 4%, medium
on 96% of the area, and 89% of the area is well represented by the sampled sites. This is considered good, 
supporting active soil biology and maintaining current practices. Recommendations: On 79% of the area (very 
strongly acid): Aluminium toxicity likely and P poorly available; liming strongly advised. On 93% of the area 
(high): Good; maintain current practices. Class ranges follow USDA Soil Survey Manual (Soil Science Division Staff 
2017) (pH) and Hazelton & Murphy (2016) Interpreting Soil Test Results (SOC).

[Step 2: Duration 6.49 seconds| Input tokens: 12,281 | Output tokens: 492]

'A farmer near Kakamega town should apply lime. Soil pH around Kakamega town: a mean of 5.0 pH units within 3 km, very strongly acid on 79% and strongly acid on 21% of the area; confidence is medium on 98%, low on 2% of the area, and 90% of the area is well represented by the sampled sites. This indicates that aluminium toxicity is likely and phosphorus is poorly available, suggesting strong liming advice. Soil organic carbon around Kakamega town: a mean of 19.2 g/kg within 3 km, high on 93% and moderate on 7% of the area; confidence is high on 4%, medium on 96% of the area, and 89% of the area is well represented by the sampled sites. This is considered good, supporting active soil biology and maintaining current practices. Recommendations: On 79% of the area (very strongly acid): Aluminium toxicity likely and P poorly available; liming strongly advised. On 93% of the area (high): Good; maintain current practices. Class ranges follow USDA Soil Survey Manual (Soil Science Division Staf

In [8]:
agent.run("Summarize soil conditions within 5 km of Malava.")

╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ Summarize soil conditions within 5 km of Malava.                                                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  malava_location = find_place(name="Malava")                                                                      
  print(malava_location)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'query': 'Malava', 'matches': [{'name': 'Malava, Kakamega, Kenya', 'lat': 0.46928, 'lon': 34.83442, 'type': 
'administrative'}, {'name': 'Malava, Chemuche ward, Malava, Kakamega, Kenya', 'lat': 0.44254, 'lon': 34.85354, 
'type': 'town'}]}

Out: None

[Step 1: Duration 2.08 seconds| Input tokens: 5,042 | Output tokens: 71]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(                                                                                                    
      f"Soil conditions within 5 km of Malava, Chemuche ward, Kakamega, Kenya: "                                   
      f"Soil pH around Malava: {area_summary['properties']['ph']['summary']}. "                                    
      f"This indicates that most of the area is acidic, which can lead to aluminium toxicity and reduced           
  phosphorus availability for crops. "                                                                             
      f"Soil organic carbon around Malava: {area_summary['properties']['soc_gkg']['summary']}. "                   
      f"This suggests good soil structure and nutrient supply, which is beneficial for crop growth. "              
      f"Recommendations: {area_summary['properties']['ph']['advice_text']}.                                        
  {area_summary['properties']['soc_gkg']['advice_text']}. "                                                        
      f"Class ranges follow the USDA Soil Survey Manual (Soil Science Division Staff 2017) (pH) and Hazelton &     
  Murphy (2016) Interpreting Soil Test Results (SOC)."                                                             
  )                                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Soil conditions within 5 km of Malava, Chemuche ward, Kakamega, Kenya: Soil pH around Malava: a mean 
of 5.1 pH units within 5 km, strongly acid on 59% and very strongly acid on 41% of the area; confidence is medium 
on 95%, low on 4% of the area, and 96% of the area is well represented by the sampled sites. This indicates that 
most of the area is acidic, which can lead to aluminium toxicity and reduced phosphorus availability for crops. 
Soil organic carbon around Malava: a mean of 18.4 g/kg within 5 km, high on 80% and moderate on 20% of the area; 
confidence is high on 1%, medium on 99% of the area, and 94% of the area is well represented by the sampled sites. 
This suggests good soil structure and nutrient supply, which is beneficial for crop growth. Recommendations: On 59%
of the area (strongly acid): Aluminium toxicity possible for sensitive crops; liming advised for maize and beans. 
On 41% of the area (very strongly acid): Aluminium toxicity likely and P poorly available; liming strongly advised.
On 80% of the area (high): Good; maintain current practices. Class ranges follow the USDA Soil Survey Manual (Soil 
Science Division Staff 2017) (pH) and Hazelton & Murphy (2016) Interpreting Soil Test Results (SOC).

[Step 3: Duration 5.29 seconds| Input tokens: 16,859 | Output tokens: 425]

'Soil conditions within 5 km of Malava, Chemuche ward, Kakamega, Kenya: Soil pH around Malava: a mean of 5.1 pH units within 5 km, strongly acid on 59% and very strongly acid on 41% of the area; confidence is medium on 95%, low on 4% of the area, and 96% of the area is well represented by the sampled sites. This indicates that most of the area is acidic, which can lead to aluminium toxicity and reduced phosphorus availability for crops. Soil organic carbon around Malava: a mean of 18.4 g/kg within 5 km, high on 80% and moderate on 20% of the area; confidence is high on 1%, medium on 99% of the area, and 94% of the area is well represented by the sampled sites. This suggests good soil structure and nutrient supply, which is beneficial for crop growth. Recommendations: On 59% of the area (strongly acid): Aluminium toxicity possible for sensitive crops; liming advised for maize and beans. On 41% of the area (very strongly acid): Aluminium toxicity likely and P poorly available; liming str

In [9]:
agent.run("What pH values count as strongly acid, and why does it matter for maize?")

╭──────────────────────────────────────────── New run - soil_advisor ─────────────────────────────────────────────╮
│                                                                                                                 │
│ What pH values count as strongly acid, and why does it matter for maize?                                        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  class_def = get_class_definitions(property_name='ph', class_name='strongly acid')                                
  print(class_def)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'ph': {'label': 'Soil pH (water)', 'class_ranges_source': 'USDA Soil Survey Manual (Soil Science Division Staff 
2017)', 'interpretation_source': 'project indicator table (config/indicator_thresholds.csv)', 'classes': [{'class':
'Strongly acid', 'range': '5.1 to <5.6 pH units', 'summary': 'Soil pH (water) values of 5.1 to <5.6 pH units count 
as strongly acid', 'advice_text': 'Aluminium toxicity possible for sensitive crops; liming advised for maize and 
beans', 'interpretation': 'Aluminium toxicity possible for sensitive crops; liming advised for maize and beans', 
'explanation': 'Exchangeable aluminium may be present and can limit root growth of sensitive crops such as maize 
and beans; phosphorus availability is reduced; raising pH improves fertilizer use efficiency'}]}}

Out: None

[Step 1: Duration 2.10 seconds| Input tokens: 5,045 | Output tokens: 77]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  ph_summary = class_def['ph']['classes'][0]['summary']                                                            
  ph_advice_text = class_def['ph']['classes'][0]['advice_text']                                                    
  ph_explanation = class_def['ph']['classes'][0]['explanation']                                                    
  ph_class_ranges_source = class_def['ph']['class_ranges_source']                                                  
                                                                                                                   
  final_answer(f"Soil pH values of 5.1 to <5.6 pH units count as strongly acid. This indicates that the soil may   
  contain high levels of aluminium, which can be toxic to sensitive crops like maize. Liming is advised for maize  
  and beans to mitigate aluminium toxicity and improve phosphorus availability and fertilizer use efficiency.      
  Class ranges follow the USDA Soil Survey Manual (Soil Science Division Staff 2017).")                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Soil pH values of 5.1 to <5.6 pH units count as strongly acid. This indicates that the soil may 
contain high levels of aluminium, which can be toxic to sensitive crops like maize. Liming is advised for maize and
beans to mitigate aluminium toxicity and improve phosphorus availability and fertilizer use efficiency. Class 
ranges follow the USDA Soil Survey Manual (Soil Science Division Staff 2017).

[Step 2: Duration 4.25 seconds| Input tokens: 10,439 | Output tokens: 272]

'Soil pH values of 5.1 to <5.6 pH units count as strongly acid. This indicates that the soil may contain high levels of aluminium, which can be toxic to sensitive crops like maize. Liming is advised for maize and beans to mitigate aluminium toxicity and improve phosphorus availability and fertilizer use efficiency. Class ranges follow the USDA Soil Survey Manual (Soil Science Division Staff 2017).'